The following code checks the compute instance type to ensure there are enough compute resources to run this project.

In [ ]:
def verify_colab_runtime():
    import os
    import psutil
    import torch

    cpu_count = os.cpu_count()
    ram_gb = psutil.virtual_memory().total / 1024**3

    print(f"CPUs: {cpu_count}")
    print(f"RAM: {ram_gb:.1f} GB")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print("Colab GPU verified ✓")
    else:
        print("No GPU found.")
        print("Fix: Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save")

verify_colab_runtime()

CPUs: 12
RAM: 53.0 GB
GPU: NVIDIA L4
Colab GPU verified ✓


LLM & Datasets required packages

In [ ]:
!pip install --no-cache-dir -U transformers accelerate datasets evaluate rouge_score peft
!pip install -U torchao

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 403.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 413.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 315.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 391.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=dd4688398475053a369d0789ab64d9a19bd644f65dd65088e3685793200beae2
  Stored in directory: /tmp/pip-ephem-wheel-cache-eap8g3g_/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  

In [ ]:
# First upgrade pip
##%pip install --upgrade pip

# Install torch and torchdata
#%pip install --no-deps torch==2.5.1 torchdata==0.6.0 --quiet

# Then install other packages except TRL
#%pip install -U \
    #datasets==2.17.0 \
    #transformers==4.38.2 \
    #accelerate==0.28.0 \
    #evaluate==0.4.0 \
    #rouge_score==0.1.2 \
    #peft==0.3.0 --quiet

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np


import transformers
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.11.0+cu128
transformers: 5.11.0
accelerate: 1.13.0
cuda available: True


###**Load dataset and LLM**

I am  going to continue experimenting with the DataScienceInterviewQuestions  Hugging Face dataset https://huggingface.co/datasets/mjphayes/machine_learning_questions/viewer. It contains 636 questions with answers.

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict, Dataset
import pandas as pd

# Dataset 1: mjphayes — already lowercase question / answer
ds1 = load_dataset("mjphayes/machine_learning_questions")
if "__index_level_0__" in ds1["train"].column_names:
    ds1 = ds1.remove_columns("__index_level_0__")

# Dataset 2: UdayG01 — capitalized, so rename to lowercase
ds2 = load_dataset("UdayG01/DataScienceInterviewQuestions")
ds2 = ds2.rename_columns({"Question": "question", "Answer": "answer"})

keep = ["question", "answer"]

# Use BOTH splits of mjphayes (train 508 + test 128 = 636)
ds1_all = concatenate_datasets([
    ds1["train"].select_columns(keep),
    ds1["test"].select_columns(keep),
])

# UdayG01 train (47)
ds2_train = ds2["train"].select_columns(keep)

# Merge into one pool (636 + 47 = 683)  <-- this line was missing
combined = concatenate_datasets([ds1_all, ds2_train])
print(combined)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/656 [00:00<?, ?B/s]

data/train-00000-of-00001-7ebb9cdef03dd9(…):   0%|          | 0.00/65.7k [00:00<?, ?B/s]

data/test-00000-of-00001-fbd3905b045b12b(…):   0%|          | 0.00/20.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/508 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/304 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.68k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer'],
    num_rows: 683
})


In [ ]:
# Remove duplicate questions BEFORE splitting (avoids leakage)
df = combined.to_pandas()
before = len(df)
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)
print(f"removed {before - len(df)} duplicate questions, {len(df)} remain")
combined = Dataset.from_pandas(df, preserve_index=False)

removed 32 duplicate questions, 651 remain


In [ ]:
split_1 = combined.train_test_split(test_size=0.30, seed=42)        # 70% train
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)  # 15% val, 15% test

dataset = DatasetDict({
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"],
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 455
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 98
    })
})


In [ ]:
#huggingface_dataset_name = "mjphayes/machine_learning_questions"

#dataset = load_dataset(huggingface_dataset_name)

#dataset

Next I will load two key parts from Hugging Face: the **FLAN-T5** model and the **tokenizer**. The tokenizer turns your Q/A text into numbers the model can understand, and the model uses those numbers to generate an answer.

The LLM idea here is “starting from a pre-trained model” instead of building one from scratch. I use this when I want a model that already understands language, but not when I need a very specialized model with no useful pre-trained base.

In this Q/A data science interview project  this gives me the starting FLAN-T5 model before fine-tuning it.

In [ ]:
model_name = 'google/flan-t5-base'

# Pick GPU if available, otherwise CPU. Every model/tensor is moved to this device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# `dtype` replaces the deprecated `torch_dtype` argument.
# my original model
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, dtype=torch.bfloat16)
original_model = original_model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that.

In [ ]:
def print_number_of_trainable_model_parameters(model):
  trainable_model_params = 0
  all_model_params = 0
  for _, param in model.named_parameters():
    all_model_params += param.numel()
    if param.requires_grad:
      trainable_model_params += param.numel()
  return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
all model parameters: 247577856
percentage of trainable model parameters: 100.00%


This means the model has 247,577,856 total settings/knobs inside it.
Because 247,577,856 are trainable, that means the project is allowing every single knob in the model to be updated during training.

So 100% trainable means this is **full fine-tuning**, not PEFT/LoRA. I am going to let the whole **FLAN-T5** model learn from the Question & Answer examples.

###**Test the Model with Zero Shot Inferencing**

You can see that the model struggles to answer the question compared to the baseline human answer, but it does pull out some important information from the text which indicates the model can be fine-tuned to the task at hand.

In [ ]:
index = 60
question = dataset['test'][index]['question']
answer = dataset['test'][index]['answer']

prompt = f"""
Answer the following data science interview question clearly and concisely.

{question}

Answer:
"""
inputs = tokenizer(prompt, return_tensors='pt').to(device)
output = tokenizer.decode(
    original_model.generate(
        inputs["input_ids"],
        max_new_tokens=100,
    )[0],
    skip_special_tokens=True
)


dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{answer}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Answer the following data science interview question clearly and concisely.

How is the accuracy_score_v2 function calculated?

Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
The accuracy_score_v2 function is calculated using the formula (TP + TN) / (TP + TN + FP + FN), where TP is true positives, TN is true negatives, FP is false positives, and FN is false negatives【31†source】.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
accuracy_score_v2 function


This **baseline model** is showing that the raw FLAN-T5 model understands the topic a little, but it is giving a weak, surface-level answer. It says boosting is “a method of boosting,” which is basically repeating the word instead of explaining the real difference between bagging and boosting.

A possible reason is that the model has not been fine-tuned on the data science interview answer style yet, so it does not know that I expect a clear, complete interview explanation. This is the **before** result you will compare against after full fine-tuning or PEFT.

###**Perform Full Fine Tuning**

Before fine-tuning, you must turn each row in your dataset into a clear instruction format so the model knows what task it is learning.

For the data science interview dataset:
- Question = the input text
- Answer = the correct output

So, the training example should look like this:

Training prompt:

- `Answer the following data science interview question.`

- `What is the difference between bagging and boosting in machine learning?`

Answer:
Training response:

- `Bagging and boosting are ensemble learning techniques that improve model performance by combining multiple models...`

Then I would convert both the prompt and the correct answer into tokens. Tokens are just the numbers the model uses to read text and learn from it.

I am teaching **FLAN-T5**, `When I give you a data science interview question, answer it in this correct style.`

`input_ids` = the interview question prompt the model reads

`labels` = the correct answer the model should learn to generate

In [ ]:
def tokenize_function(example):
    # This creates the instruction the model will read before each question.
    start_prompt = 'Answer the following data science interview question clearly and concisely.\n\n'
    end_prompt = '\n\nAnswer: '

    # Because batched=True, example["question"] is a list of questions.
    # This builds one full prompt for each question.
    prompt = [start_prompt + question + end_prompt for question in example["question"]]

    # Tokenize the prompts (model inputs). Cap the length explicitly.
    model_inputs = tokenizer(
        prompt,
        max_length=512,
        padding="max_length",
        truncation=True,
    )

    # Tokenize the answers (labels) the model should learn to generate.
    labels = tokenizer(
        text_target=example["answer"],
        max_length=512,
        padding="max_length",
        truncation=True,
    )

    # Replace pad-token ids in the labels with -100 so they are IGNORED by the
    # loss. Without this the model is penalised for predicting padding and the
    # training loss explodes (this was the cause of the ~42 loss earlier).
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
        for seq in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# The tokenize_function handles all rows in batches.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Remove the original text columns; keep input_ids and labels for the model.
tokenized_datasets = tokenized_datasets.remove_columns(['question', 'answer'])

Map:   0%|          | 0/455 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

In [ ]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

print(tokenized_datasets)

Shapes of the datasets:
Training: (455, 3)
Validation: (98, 3)
Test: (98, 3)
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 455
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 98
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 98
    })
})


###**Fine-Tune the Model with the Preprocessed Dataset**

This part uses Hugging Face’s Trainer, which is like a built-in training manager. Instead of writing the whole training loop, I give it the model, the tokenized dataset, and the training settings, then it handles the fine-tuning process me. The extra training settings were chosen by trial and error, so right now I do not need to understand every detail. Just understand that this cell is where the model starts learning from the prepared Question and Answer examples.

Its taking my pre-trained FLAN-T5 model and my tokenized data science interview questions, then train the model to produce answers that look like the human Answer column.

In [ ]:
output_dir = f'./flan-ds-interview-training-{str(int(time.time()))}'

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=3e-4,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

In [ ]:
#!pip uninstall -y torch torchvision torchaudio
#!pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
#!pip install --no-cache-dir -U transformers datasets accelerate evaluate rouge_score peft

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.431250,2.264668
2,2.365625,2.216518
3,2.381250,2.197704
4,2.246875,2.196110
5,1.945312,2.194515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=285, training_loss=2.3175438596491227, metrics={'train_runtime': 193.5941, 'train_samples_per_second': 11.751, 'train_steps_per_second': 1.472, 'total_flos': 1557822976819200.0, 'train_loss': 2.3175438596491227, 'epoch': 5.0})

Training a fully fine-tuned version of the model would take a few hours on a GPU. To save time, download a checkpoint of the fully fine-tuned model to use in the rest of this notebook. This fully fine-tuned model will also be referred to as the instruct model for this project.

In [ ]:

#!aws s3 cp --recursive s3://dlai-generative-ai/models/flan-dialogue-summary-checkpoint/ ./flan-dialogue-summary-checkpoint/
#!ls -alh ./flan-dialogue-summary-checkpoint/pytorch_model.bin

In [ ]:
#Create an instance of the AutoModelForSeq2SeqLM class for the instruct model:
#instruct_model = AutoModelForSeq2SeqLM.from_pretrained("./flan-dialogue-summary-checkpoint", torch_dtype=torch.bfloat16)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
trainer.save_model("/content/drive/MyDrive/flan-ds-interview-checkpoint")
tokenizer.save_pretrained("/content/drive/MyDrive/flan-ds-interview-checkpoint")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/flan-ds-interview-checkpoint/tokenizer_config.json',
 '/content/drive/MyDrive/flan-ds-interview-checkpoint/tokenizer.json')

### Saving the Fine-Tuned Model

After training, the model only exists in memory. These two lines save it to a folder so it isn't lost:

- **`trainer.save_model(...)`** → saves the trained model (the weights it just learned).
- **`tokenizer.save_pretrained(...)`** → saves the matching tokenizer (turns text into numbers the model understands).

**Why it matters:**

1. **Don't lose your work** — without saving, closing or disconnecting the notebook erases the trained model and you'd have to retrain.
2. **The next cell needs it** — the notebook reloads this folder to create `instruct_model` and compare it against the original.
3. **Reuse anytime** — once saved, you can reload the model later without training again.

> ⚠️ **Colab note:** This saves to temporary storage that disappears when the runtime disconnects. To keep it permanently, mount Google Drive and save there instead.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

instruct_model = AutoModelForSeq2SeqLM.from_pretrained("/content/drive/MyDrive/flan-ds-interview-checkpoint")
instruct_model = instruct_model.to(device)
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/flan-ds-interview-checkpoint")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# Sanity check that the fine-tuned model is on the expected device.
print("instruct_model device:", next(instruct_model.parameters()).device)
print("target device:", device)

instruct_model device: cuda:0
target device: cuda


###**Evaluate the Model Qualitatively (Human Evaluation)**

As with many GenAI applications, a qualitative approach where you ask yourself the question "Is my model behaving the way it is supposed to?" is usually a good starting point. In the example below (the same one we started this notebook with), you can see how the fine-tuned model is able to create a reasonable answer of the question compared to the original inability to understand what is being asked of the model.

In [ ]:
index = 12
question = dataset['test'][index]['question']
human_baseline_answer = dataset['test'][index]['answer']

prompt = f"""
Answer the following data science interview question clearly and concisely.

{question}

Answer:
"""

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

print(f'QUESTION:\n{question}')
print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{human_baseline_answer}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(dash_line)
print(f'INSTRUCT MODEL:\n{instruct_model_text_output}')

QUESTION:
What is the role of 'optimization algorithms' in machine learning?
---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Optimization algorithms in machine learning are used to minimize or maximize a function, which is often the loss function used to train a model.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
Optimization algorithms in machine learning are used to optimize the performance of a model by minimizing the number of errors in the model, thereby minimizing the number of errors in the model.
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
Optimization algorithms in machine learning are used to optimize the performance of a model by minimizing the number of errors in the model, thereby minimizing the number of errors in the model.


###**Evaluate the Model Qualitatively (Human Evaluation)**
As with many GenAI applications, a qualitative approach where you ask yourself the question "Is my model behaving the way it is supposed to?" is usually a good starting point. In the example below (the same one we started this notebook with), you can see how the fine-tuned model is able to create a reasonable summary of the Q/A compared to the original inability to understand what is being asked of the model.

In [ ]:
index = 18
question = dataset['test'][index]['question']
human_baseline_answer = dataset['test'][index]['answer']

prompt = f"""
Answer the following data science interview question clearly and concisely.


{question}

Answer:
"""

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

print(f'QUESTION:\n{question}')
print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{human_baseline_answer}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(dash_line)
print(f'INSTRUCT MODEL:\n{instruct_model_text_output}')


QUESTION:
### Question:
What is the curse of dimensionality?

### Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
The curse of dimensionality refers to various phenomena that arise when analyzing and organizing data in high-dimensional spaces...
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
The curse of dimensionality is that it causes the dimensionality of a data set to be arbitrary, causing it to be a dimensionality of the data set.
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
The curse of dimensionality is that it causes the dimensionality of a data set to be arbitrary, causing it to be a dimensionality of the data set.


###**Evaluate the Model Quantitatively (with ROUGE Metric)**

The ROUGE metric helps quantify the validity of Q/A produced by models. It compares answers to a "baseline" answer which is usually created by a human. While not perfect, it does indicate the overall increase in answer effectiveness that we have accomplished by fine-tuning.

In [ ]:
rouge = evaluate.load('rouge')

In [ ]:
#Generate the outputs for the sample of the test dataset (only 10 dialogues and summaries to save time), and save the results.

# --- STEP A: reload a FRESH, untrained baseline ---
original_model = AutoModelForSeq2SeqLM.from_pretrained(
    'google/flan-t5-base', dtype=torch.bfloat16
).to(device)


questions = dataset['test'][0:10]['question']
human_baseline_answer = dataset['test'][0:10]['answer']

original_model_answer = []
instruct_model_answer = []

gen_config = GenerationConfig(max_new_tokens=200, num_beams=1, no_repeat_ngram_size=3)

for q in questions:                      # q = ONE question per loop
    prompt = f"""
    Answer the following data science interview question clearly and concisely.


{q}

Answer: """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    orig_out = original_model.generate(input_ids=input_ids, generation_config=gen_config)
    original_model_answer.append(tokenizer.decode(orig_out[0], skip_special_tokens=True))

    inst_out = instruct_model.generate(input_ids=input_ids, generation_config=gen_config)
    instruct_model_answer.append(tokenizer.decode(inst_out[0], skip_special_tokens=True))

zipped_answers = list(zip(human_baseline_answer, original_model_answer, instruct_model_answer))
df = pd.DataFrame(zipped_answers, columns=['human_baseline_answer', 'original_model_answer', 'instruct_model_answer'])
df

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,human_baseline_answer,original_model_answer,instruct_model_answer
0,K-nearest neighbors (KNN) in machine learning ...,a neural network,KNN in machine learning is a technique where a...
1,Collaborative filtering in machine learning is...,a method of detecting and evaluating a large n...,Collaborative filtering in machine learning is...
2,Unsupervised datasets can be visualized by tec...,Using a computer program,Unsupervised datasets can be visualized by usi...
3,An orthonormal basis in linear algebra is a ba...,It is a measure of the degree to which the und...,An orthonormal basis in linear algebra is a co...
4,Time series analysis in machine learning invol...,time series analysis,Time series analysis in machine learning is a ...
5,Precision in machine learning is a metric that...,precision,Precision in machine learning is the ability t...
6,The span of a set of vectors is the set of all...,a set of vectors,The span of a set of vectors is the number of ...
7,Batch normalization in deep learning is a tech...,a method for detecting a large number of error...,Batch normalization in deep learning is a tech...
8,Logistic regression in classification predicts...,a statistical method for estimating the likeli...,Logistic regression is a method used to estima...
9,"In the exponential model, the best approximati...",The exponential model is a continuous linear m...,The exponential model is used to learn Bayesia...


In [ ]:
#Evaluate the models computing ROUGE metrics. Notice the improvement in the results!

original_model_results = rouge.compute(
    predictions=original_model_answer,
    references=human_baseline_answer[0:len(original_model_answer)],
    use_aggregator=True,
    use_stemmer=True,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_answer,
    references=human_baseline_answer[0:len(instruct_model_answer)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('INSTRUCT MODEL:')
print(instruct_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.21281159654185972), 'rouge2': np.float64(0.10681815200996785), 'rougeL': np.float64(0.1812268360294676), 'rougeLsum': np.float64(0.18519394129262548)}
INSTRUCT MODEL:
{'rouge1': np.float64(0.451071676751971), 'rouge2': np.float64(0.26016160804347993), 'rougeL': np.float64(0.3826555304014881), 'rougeLsum': np.float64(0.38402234992806494)}


In [ ]:
print("Improvement of INSTRUCT over ORIGINAL:")
for key in original_model_results:
    base = original_model_results[key]
    diff = (instruct_model_results[key] - base) / base * 100
    print(f'{key}: {diff:+.1f}%')

Improvement of INSTRUCT over ORIGINAL:
rouge1: +112.0%
rouge2: +143.6%
rougeL: +111.1%
rougeLsum: +107.4%


###**Perform Parameter Efficient Fine-Tuning (PEFT)**

Now, let's perform Parameter Efficient Fine-Tuning (PEFT) fine-tuning as opposed to "full fine-tuning" as I did above. PEFT is a form of instruction fine-tuning that is much more efficient than full fine-tuning - with comparable evaluation results.

PEFT is a generic term that includes Low-Rank Adaptation (LoRA) and prompt tuning (which is NOT THE SAME as prompt engineering!). In most cases, when someone says PEFT, they typically mean LoRA. LoRA, at a very high level, allows the user to fine-tune their model using fewer compute resources (in some cases, a single GPU). After fine-tuning for a specific task, use case, or tenant with LoRA, the result is that the original LLM remains unchanged and a newly-trained “LoRA adapter” emerges. This LoRA adapter is much, much smaller than the original LLM - on the order of a single-digit % of the original LLM size (MBs vs GBs).

That said, at inference time, the LoRA adapter needs to be reunited and combined with its original LLM to serve the inference request. The benefit, however, is that many LoRA adapters can re-use the original LLM which reduces overall memory requirements when serving multiple tasks and use cases.


###**Setup the PEFT/LoRA model for Fine-Tuning**

I need to set up the PEFT/LoRA model for fine-tuning with a new layer/parameter adapter. Using PEFT/LoRA, you are freezing the underlying LLM and only training the adapter. Have a look at the LoRA configuration below. Note the rank (r) hyper-parameter, which defines the rank/dimension of the adapter to be trained.

Does LoRA reach similar quality while training far fewer parameters?

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32, # Rank
    lora_alpha=32,
    target_modules=["q","v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

In [ ]:
# Add LoRA adapter layers/parameters to the original LLM to be trained.

peft_model = get_peft_model(original_model,
                            lora_config)
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%


###**Train PEFT Adapter**
Define training arguments and create Trainer instance.

In [ ]:
output_dir = f'./flan-ds-interview-training-{str(int(time.time()))}'


peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=2e-3, # Higher learning rate than full fine-tuning.
    num_train_epochs=3,
    logging_steps=5,
    eval_strategy="epoch", #see validation loss
    report_to="none",
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets['validation'],
)

In [ ]:
#Add eval_strategy="epoch" this time — that's how you catch overfitting:
#watch for training loss falling while validation loss rises.
#The moment validation loss turns up, that's your overfitting point, and you want fewer epochs than that.

peft_trainer.train()


Epoch,Training Loss,Validation Loss
1,2.450000,2.286352
2,2.387500,2.230230
3,2.468750,2.222258


TrainOutput(global_step=171, training_loss=2.490451388888889, metrics={'train_runtime': 111.8393, 'train_samples_per_second': 12.205, 'train_steps_per_second': 1.529, 'total_flos': 949533569187840.0, 'train_loss': 2.490451388888889, 'epoch': 3.0})

In [ ]:
peft_model_path = "./peft-ds-interview-checkpoint-local"

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

('./peft-ds-interview-checkpoint-local/tokenizer_config.json',
 './peft-ds-interview-checkpoint-local/tokenizer.json')

In [ ]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM
import torch

# 1) Load base model AND move it to GPU first
base = AutoModelForSeq2SeqLM.from_pretrained(
    'google/flan-t5-base', dtype=torch.bfloat16
).to(device)

# 2) Attach the adapter to the already-on-GPU base
peft_model = PeftModel.from_pretrained(base, "./peft-ds-interview-checkpoint-local")

# 3) Move the whole thing again to be safe
peft_model = peft_model.to(device)

# 4) Verify INSIDE the same cell
print("base:", next(base.parameters()).device)
print("peft:", next(peft_model.parameters()).device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


base: cuda:0
peft: cuda:0


Prepare this model by adding an adapter to the original FLAN-T5 model. You are setting is_trainable=False because the plan is only to perform inference with this PEFT model. If you were preparing the model for further training, you would set is_trainable=True.


In [ ]:
from peft import PeftModel, PeftConfig

peft_model_base = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", dtype=torch.bfloat16).to(device)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

peft_model = PeftModel.from_pretrained(
    peft_model_base,
    './peft-ds-interview-checkpoint-local',
    is_trainable=False,
).to(device)

print("peft:", next(peft_model.parameters()).device)   # should say cuda:0

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


peft: cuda:0


In [ ]:
#The number of trainable parameters will be 0 due to is_trainable=False setting:

print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 0
all model parameters: 251116800
percentage of trainable model parameters: 0.00%


###**Evaluate the Model Qualitatively (Human Evaluation)**

Make inferences for the same example a with the original model, fully fine-tuned and PEFT model.

In [ ]:
index = 50
question = dataset['test'][index]['question']
human_baseline_answer = dataset['test'][index]['answer']

prompt = f"""
Answer the following data science interview question clearly and concisely.

{question}

Answer: """

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

print(f'QUESTION:\n{question}')
print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{human_baseline_answer}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(dash_line)
print(f'INSTRUCT MODEL:\n{instruct_model_text_output}')
print(dash_line)
print(f'PEFT MODEL: {peft_model_text_output}')

QUESTION:
Explain the concept of vector addition.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Vector addition is the operation of adding two vectors together, by adding their corresponding components.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
Vector addition is a method of adding a vector to a set of vectors, a technique that is used to add a vector to a set of vectors.
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
Vector addition is a technique where a vector is added to a vector to make it more complex.
---------------------------------------------------------------------------------------------------
PEFT MODEL: Vector addition is a method of adding a vector to a set of vectors, a technique that is used to add a vector to a set of vectors.


###**Evaluate the Model Quantitatively (with ROUGE Metric)**

Perform inferences for the sample of the test dataset (using the full dataset instead of 10).

Reason:  a score from 10 examples is shaky; a score from hundreds is trustworthy. Same technique, bigger sample. I want to show that the conclusion ("fine-tuning helps") holds up when measured properly, not just on a lucky 10.

In [ ]:
# Fresh baseline for the comparison
original_model = AutoModelForSeq2SeqLM.from_pretrained(
    'google/flan-t5-base', dtype=torch.bfloat16
).to(device)


questions = dataset['test']['question']
human_baseline_answers = dataset['test']['answer']

original_model_answers = []
instruct_model_answers = []
peft_model_answers = []

for idx, q in enumerate(questions):
    prompt = f"""
Answer the following data science interview question clearly and concisely.

{q}

Summary: """

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    human_baseline_text_output = human_baseline_answers[idx]

    original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

    instruct_model_outputs = instruct_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    instruct_model_text_output = tokenizer.decode(instruct_model_outputs[0], skip_special_tokens=True)

    peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

    original_model_answers.append(original_model_text_output)
    instruct_model_answers.append(instruct_model_text_output)
    peft_model_answers.append(peft_model_text_output)

zipped_answers = list(zip(human_baseline_answers, original_model_answers, instruct_model_answers, peft_model_answers))

df = pd.DataFrame(zipped_answers, columns = ['human_baseline_answers', 'original_model_answers', 'instruct_model_answers', 'peft_model_answers'])
df

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,human_baseline_answers,original_model_answers,instruct_model_answers,peft_model_answers
0,K-nearest neighbors (KNN) in machine learning ...,a neural network,KNN in machine learning is a technique used to...,KNN in machine learning is a method for estima...
1,Collaborative filtering in machine learning is...,a machine learning algorithm that combines the...,Collaborative filtering in machine learning is...,Collaborative filtering in machine learning is...
2,Unsupervised datasets can be visualized by tec...,Use a computer program to visualize the data.,Unsupervised datasets can be visualized by usi...,Unsupervised datasets can be visualized by com...
3,An orthonormal basis in linear algebra is a ba...,"In linear algebra, the orthonormal basis is th...",An orthonormal basis in linear algebra is a co...,An orthonormal basis in linear algebra is a co...
4,Time series analysis in machine learning invol...,time series analysis,Time series analysis in machine learning is a ...,Time series analysis in machine learning is a ...
...,...,...,...,...
93,"Continuous variables, like height, have a cont...",The continuous variable is the variable that i...,Continuous variables are not discrete variable...,Continuous variables are not defined as a cont...
94,Clustering in machine learning is the task of ...,Clustering is the process of combining a set o...,Clustering in machine learning is a technique ...,Clustering in machine learning is a technique ...
95,"No, the standard product of two matrices is no...","Answer the question with a clear, concise, and...","Yes, the standard product of two matrices is a...",The standard product of two matrices is a matr...
96,An intelligence explosion refers to the idea t...,a computer program that can be programmed to l...,An intelligence explosion in AI is a process w...,An intelligence explosion in AI is a process w...


In [ ]:
#Compute ROUGE score for this subset of the data.
rouge = evaluate.load('rouge')

original_model_results = rouge.compute(
    predictions=original_model_answers,
    references=human_baseline_answers[0:len(original_model_answers)],
    use_aggregator=True,
    use_stemmer=True,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_answers,
    references=human_baseline_answers[0:len(instruct_model_answers)],
    use_aggregator=True,
    use_stemmer=True,
)

peft_model_results = rouge.compute(
    predictions=peft_model_answers,
    references=human_baseline_answers[0:len(peft_model_answers)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('INSTRUCT MODEL:')
print(instruct_model_results)
print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.25615197161118697), 'rouge2': np.float64(0.0860587417908808), 'rougeL': np.float64(0.20818965211677595), 'rougeLsum': np.float64(0.20865712182489343)}
INSTRUCT MODEL:
{'rouge1': np.float64(0.3715536871057331), 'rouge2': np.float64(0.1878793481025516), 'rougeL': np.float64(0.3308073950873932), 'rougeLsum': np.float64(0.33095380811069275)}
PEFT MODEL:
{'rouge1': np.float64(0.3712479780726236), 'rouge2': np.float64(0.18675279832431277), 'rougeL': np.float64(0.330834639498517), 'rougeLsum': np.float64(0.3305136086214946)}


In [ ]:
#Calculate the improvement of PEFT over the original model:

print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')


Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL
rouge1: 11.51%
rouge2: 10.07%
rougeL: 12.26%
rougeLsum: 12.19%


In [ ]:
# Now calculate the improvement of PEFT over a full fine-tuned model:

print("Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(instruct_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL
rouge1: -0.03%
rouge2: -0.11%
rougeL: 0.00%
rougeLsum: -0.04%


Here you see a small percentage decrease in the ROUGE metrics vs. full fine-tuned. However, the training requires much less computing and memory resources (often just a single GPU).

###**Results**

I compared four sets of answers across the full **98-question** held-out test set:

- **Human baseline** — the correct, expert answers (the gold standard).
- **Original model** — plain FLAN-T5-base, no fine-tuning.
- **Instruct model** — my full fine-tuned model (all parameters).
- **PEFT model** — my LoRA fine-tuned model (lightweight; ~1% of parameters trained).

**ROUGE scores:**

| Metric | Original | Instruct (full) | PEFT (LoRA) |
|--------|----------|-----------------|-------------|
| rouge1 | 0.256    | 0.372           | 0.371 |
| rouge2 | 0.086    | 0.188           | 0.187 |
| rougeL | 0.208    | 0.331           | 0.331 |
| rLsum  | 0.209    | 0.331           | 0.331 |

**What I observed:**

1. **Fine-tuning clearly helped.** The original model often gave vague or off-topic answers
   (e.g. "a neural network" for K-nearest neighbors). Both fine-tuned models give real, on-topic
   answers that begin defining the concept correctly. Full fine-tuning improved rouge1 by **+45%**
   (0.256 → 0.372) and more than **doubled** rouge2 (0.086 → 0.188).

2. **PEFT matched full fine-tuning almost exactly.** PEFT scored within **0.1%** of the full
   fine-tune on every metric (e.g. rougeL 0.3308 vs 0.3309). This is the key result: **LoRA
   achieved the same quality while training only ~1% of the parameters** — far cheaper and with
   a checkpoint a fraction of the size.

3. **Answers are still short and somewhat incomplete.** The models reliably name the right topic
   and start a correct definition, but rarely give the full explanation a human does. This is
   expected: FLAN-T5-base is a small model, the dataset is small, and the training answers
   themselves are brief — so the models learned a concise style.

**Bottom line:** Fine-tuning moved the model from vague/off-topic answers to correct, on-topic
responses, improving ROUGE by ~45% (rouge1) to over 100% (rouge2). PEFT/LoRA delivered essentially
identical quality to full fine-tuning at a tiny fraction of the training cost — demonstrating that
parameter-efficient fine-tuning is a highly effective alternative to full fine-tuning.

###**Fixes to Improve Results**

Use a bigger base model. FLAN-T5-base (250M) is small. Switch to flan-t5-large (780M) for noticeably more complete answers. Biggest single improvement.

Train longer on more data. Use the full ~650-row dataset, 3–5 epochs, no max_steps cap. More exposure = better answers.

Always use anti-repetition decoding so you never get the "a set of a set of..." loops: GenerationConfig(max_new_tokens=200, no_repeat_ngram_size=3, repetition_penalty=1.3)

Match the prompt in training and inference — use the exact same wording in both, or the model underperforms.

Add more quality Q&A data. The single most effective fix long-term: more clean, complete, on-topic examples teach the model to give complete answers.

In [ ]:
# Add this at the end of Lab 2 — save to Drive, not just local
from google.colab import drive
drive.mount("/content/drive")

drive_save_path = "/content/drive/MyDrive/peft-ds-interview-checkpoint-local"

peft_trainer.model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print("Saved to Drive:", drive_save_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive: /content/drive/MyDrive/peft-ds-interview-checkpoint-local


In [ ]:
# Run this immediately after the save cell completes
import os

drive_save_path = "/content/drive/MyDrive/peft-ds-interview-checkpoint-local"

expected_files = [
    "adapter_config.json",
    "adapter_model.safetensors",  # or adapter_model.bin depending on version
    "tokenizer_config.json",
    "tokenizer.json",
]

print("Verifying checkpoint files:")
for f in expected_files:
    full_path = os.path.join(drive_save_path, f)
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    print(f"  {f}: {'✓' if exists else 'MISSING'} ({size:,} bytes)")

Verifying checkpoint files:
  adapter_config.json: ✓ (1,013 bytes)
  adapter_model.safetensors: ✓ (14,176,016 bytes)
  tokenizer_config.json: ✓ (2,381 bytes)
  tokenizer.json: ✓ (2,424,069 bytes)


The  checkpoint saved correctly. All four files are present and none are zero bytes.

The one file worth noting is adapter_model.safetensors at 14.2 MB. That is the expected size for a LoRA adapter on flan-t5-base with rank 32 targeting q and v — roughly 1% of the full 247M parameter model. That number is consistent with what this project's trainable parameter count reported.

We are now clear to run project part 3.